# Federated Statistics for Financial Fraud Detection
This notebook demonstrates how to compute federated statistics on financial transaction data using NVIDIA FLARE (NVFlare). It shows how to define statistical recipes, execute them in a federated environment, and visualize the results from different distributed sites.

## 1. Define Stats Recipe

In [1]:
from src.stats.client import FinancialStatistics

from nvflare.recipe.fedstats import FedStatsRecipe
from nvflare.recipe.prod_env import ProdEnv

df_stats_generator = FinancialStatistics(
    data_path="/workspace/dataset/paysim1/PS_20174392719_1491204439457_log.csv", # assumes each client uses the same data path
    data_features=['amount'], #  'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud'
) 

statistic_configs = {
    "count": {},
    "mean": {},
    "sum": {},
    "stddev": {},
    "histogram": {"*": {"bins": 128, "range": [1, 1e6]}, "amount": {"bins": 128, "range": [0, 500e3]}, "isFraud": {"bins": 2, "range": [0, 1]}},
    #"quantile": {"*": [0.1, 0.5, 0.9], "isFraud": [0.01, 0.2, 0.5]},
    "quantile": {"*": [0.5], "amount": [0.5]},
}

recipe = FedStatsRecipe(
    name="fedstats",
    sites=["site5"],
    statistic_configs=statistic_configs,
    stats_generator=df_stats_generator,
    stats_output_path="statistics/financial_stats.json"
)

# optionally export
#recipe.export("/tmp/nvflare/job_configs")

## 2. Run in Production Environment

In [2]:
env = ProdEnv(startup_kit_dir="/scratch/hroth/Code/JPM/admin")
run = recipe.execute(env=env)


Connecting to FLARE ...
Submitted job 'fed_job' with ID: 3dbccb53-4bf6-4d3d-9832-1443d121bd75


### Get Status

In [3]:
print("Job Status is:", run.get_status())

Connecting to FLARE ...
Job Status is: RUNNING


### Get Results

In [ ]:
result_path = run.get_result()
print("Result can be found in:", result_path)

Connecting to FLARE ...
job monitor done: rc=<MonitorReturnCode.JOB_FINISHED: 0>


## 3. Show stats

In [ ]:
stats_file = result_path + '/workspace/statistics/financial_stats.json'
!head -c 200 {stats_file}

In [ ]:
import json

from nvflare.app_opt.statistics.visualization.statistics_visualization import Visualization

with open(stats_file, 'r') as f:
    data = json.load(f)

### Overall stats

In [ ]:
vis = Visualization()
vis.show_stats(data = data)

## Histogram Visualization

(see [here](https://github.com/NVIDIA/NVFlare/blob/main/examples/advanced/federated-statistics/df_stats/demo/visualization.ipynb) for more display options)

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100%  depth:100% !important; }</style>"))

vis.show_histograms(data = data) # display_format = "percent"